<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>CONVOLUTION AUTOENCODER -  DIMENSIONALITY REDUCTION</strong><br>
    <strong></strong> SIVA VIKNESH<br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import time
import torch
import random
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor, '\n')

In [ ]:
# Set random seed for reproducibility
seed = 15
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
input_var    = 3                                        # INPUT VARIABLES {a(x, y), x, y}
x_dim        = 256
y_dim        = 256

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
neurons      = 128
epochs       = 300
batchsize    = 30 
in_channels  = 1                       # NO. OF INPUT CHANNELS
out_channels = 1                       # NO. OF OUTPUT CHANNELS

learn_rate_a = 1e-3
step_eph_a   = 20
decay_rate   = 0.50

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_256_0.vtk"
data_file    = "Grid_256_"
Nfile        = 1000                                 # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../../")
directory    = os.getcwd()
path         = directory + "/Flow_Data/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xscale       = 10.0
yscale       = 5.0
vort_max     = 4.4

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim))/xscale 
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim))/yscale

print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
fieldname3  = 'vort_z'
vort        = np.zeros((Nfile, x_dim, y_dim))

for i in range(Nfile):
    file_name = data_file + str(i) + ".vtk"
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader()
    reader.SetFileName(file_name)
    reader.Update()
    data = reader.GetOutput() 
    pointData = data.GetPointData().GetArray(fieldname)    
    vort    [i, :, :] = np.reshape(vtk_to_numpy(pointData), (x_dim, y_dim))
    
    print ("*"*85)
vort = vort / vort_max

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE CLASS FUNCTIONS
</div>


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.act = nn.SiLU()
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
    def forward(self, x):
        return self.act(x + self.conv2(self.act(self.conv1(x))))

class Encoder(nn.Module):
    def __init__(self, latent_c=32):
        super().__init__()
        self.level1 = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.SiLU(),
            ResidualBlock(64)
        )
        self.level2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, 2, 1),  # downsample /2
            nn.SiLU(),
            ResidualBlock(128)
        )
        self.level3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, 2, 1),  # downsample /4
            nn.SiLU(),
            ResidualBlock(256)
        )
        self.level4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, 2, 1),  # downsample /8
            nn.SiLU()
        )
        self.latent = nn.Conv2d(512, latent_c, 3, 1, 1)  # linear latent

    def forward(self, x):
        x = self.level1(x)
        x = self.level2(x)
        x = self.level3(x)
        x = self.level4(x)
        z = self.latent(x)
        return z


class Decoder(nn.Module):
    def __init__(self, latent_c=32):
        super().__init__()
        self.level4 = nn.Sequential(
            nn.ConvTranspose2d(latent_c, 512, 3, 1, 1),
            nn.SiLU()
        )
        self.level3 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # upsample ×2
            nn.SiLU(),
            ResidualBlock(256)
        )
        self.level2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # upsample ×2
            nn.SiLU(),
            ResidualBlock(128)
        )
        self.level1 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # upsample ×2
            nn.SiLU(),
            ResidualBlock(64)
        )
        self.out = nn.Conv2d(64, 1, 3, 1, 1)

    def forward(self, z):
        x = self.level4(z)
        x = self.level3(x)
        x = self.level2(x)
        x = self.level1(x)
        x = self.out(x)
        return x



def reproducible_split(dataset, train_size, test_size, seed=1):
    generator = torch.Generator().manual_seed(seed)
    return random_split(dataset, [train_size, test_size], generator=generator)

def count_parameters(model):
    total_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            total_params += param.numel()
    return total_params


In [ ]:
# MESH SPATIAL DATA
x_grid = torch.Tensor(x).reshape(1, x_dim, y_dim, 1)
y_grid = torch.Tensor(y).reshape(1, x_dim, y_dim, 1)

# FUNCTION MAPPING DATA
vort_input   = torch.Tensor(vort).to(processor)
vort_output  = torch.Tensor(vort).to(processor)

# SORTING THE TRAINING DATA
shape         = vort_input.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

vort_input_reshaped  = vort_input.reshape  (total_samples, 1, x_dim, y_dim)
vort_output_reshaped = vort_output.reshape (total_samples, 1, x_dim, y_dim)

dataset               = TensorDataset(vort_input_reshaped, vort_output_reshaped)
vort_train, vort_test = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(vort_train, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(vort_test,  batch_size=10, shuffle=False)
plot_loader    = DataLoader(dataset, batch_size=batchsize, shuffle=False)

encoder_model = Encoder ().to(processor)
decoder_model = Decoder ().to(processor)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.kaiming_normal_(m.weight, nonlinearity='linear')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

encoder_model.apply(init_weights)
decoder_model.apply(init_weights)


print("Encoder trainable parameters:", count_parameters(encoder_model))
print("Encoder trainable parameters:", count_parameters(decoder_model))

optimizer_encoder  = optim.Adam(encoder_model.parameters(), lr=learn_rate_a, betas = (0.9,0.99),eps = 10**-15)
optimizer_decoder  = optim.Adam(decoder_model.parameters(), lr=learn_rate_a, betas = (0.9,0.99),eps = 10**-15)

scheduler_encoder  = optim.lr_scheduler.StepLR(optimizer_encoder, step_size=step_eph_a, gamma=decay_rate)
scheduler_decoder  = optim.lr_scheduler.StepLR(optimizer_decoder, step_size=step_eph_a, gamma=decay_rate)

In [ ]:
Loss_Data   = torch.empty(size=(epochs+1, 1))
loss_func   = nn.MSELoss()

start_time = time.time()

for epoch in range (epochs+1):
    total_loss = 0.0
    for batch_idx, (input_data, output_data) in enumerate(train_loader):
        encoder_output_data = encoder_model (input_data)
        decoder_output_data = decoder_model (encoder_output_data)
        batch_loss          = loss_func (decoder_output_data, output_data)
        
        optimizer_encoder.zero_grad()
        optimizer_decoder.zero_grad()
        batch_loss.backward()

        with torch.no_grad():
            optimizer_encoder.step()
            optimizer_decoder.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', optimizer_encoder.param_groups[0]['lr'])
    print ("*"*85)
    
    scheduler_encoder.step()
    scheduler_decoder.step()
    
total_time = time.time() - start_time
print(f"\nTotal training wall time: {total_time/3600:.2f} hours")

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(encoder_model.state_dict(),   "CNAE_Resnet_Encoder.pt" )
torch.save(decoder_model.state_dict(),   "CNAE_Resnet_Decoder.pt" )
torch.save(Loss_Data  [0:epoch], "CNAE_Loss_Resnet.pt"  )

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        TESTING DATA
</div>


In [ ]:
# TEST ERROR OF CNO
encoder_model.eval()
decoder_model.eval()

total_loss = 0.0
for batch_idx, (input_data, output_data) in enumerate(test_loader):
    encoder_output_data = encoder_model (input_data)
    decoder_output_data = decoder_model (encoder_output_data)
    batch_loss          = loss_func (decoder_output_data, input_data)
    with torch.no_grad():
        total_loss += batch_loss.detach()  

N = batch_idx + 1
Test_Loss = total_loss/N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())